In [1]:
import pathlib
import fastdup

import numpy as np
import matplotlib.pyplot as plt
import warnings
import json
import pandas as pd

# Utility functions
Coped from https://github.com/huggingface/notebooks/blob/main/examples/segment_anything.ipynb

In [2]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(
        plt.Rectangle((x0, y0), w, h, edgecolor="green", facecolor=(0, 0, 0, 0), lw=2)
    )


def show_boxes_on_image(raw_image, boxes):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def show_points_on_image(raw_image, input_points, input_labels=None):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    if input_labels is None:
        labels = np.ones_like(input_points[:, 0])
    else:
        labels = np.array(input_labels)
    show_points(input_points, labels, plt.gca())
    plt.axis("on")
    plt.show()


def show_points_and_boxes_on_image(raw_image, boxes, input_points, input_labels=None):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    if input_labels is None:
        labels = np.ones_like(input_points[:, 0])
    else:
        labels = np.array(input_labels)
    show_points(input_points, labels, plt.gca())
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels == 1]
    neg_points = coords[labels == 0]
    ax.scatter(
        pos_points[:, 0],
        pos_points[:, 1],
        color="green",
        marker="*",
        s=marker_size,
        edgecolor="white",
        linewidth=1.25,
    )
    ax.scatter(
        neg_points[:, 0],
        neg_points[:, 1],
        color="red",
        marker="*",
        s=marker_size,
        edgecolor="white",
        linewidth=1.25,
    )


def show_masks_on_image(raw_image, masks, scores):
    if len(masks.shape) == 4:
        masks = masks.squeeze()
    if scores.shape[0] == 1:
        scores = scores.squeeze()

    nb_predictions = scores.shape[-1]
    fig, axes = plt.subplots(1, nb_predictions, figsize=(15, 15))

    for i, (mask, score) in enumerate(zip(masks, scores)):
        mask = mask.cpu().detach()
        axes[i].imshow(np.array(raw_image))
        show_mask(mask, axes[i])
        axes[i].title.set_text(f"Mask {i+1}, Score: {score.item():.3f}")
        axes[i].axis("off")
    plt.show()

In [3]:
SPLIT_MAP = dict(train="train", val="val", test="test")


def process_images(data: dict):
    df = pd.json_normalize(data["images"])
    df = df[["id", "file_name"]]
    df = df.rename(columns={"id": "image_id", "file_name": "filename"})
    return df


def process_annotations(data: dict):
    df = pd.json_normalize(data["annotations"])
    df = df[["image_id", "bbox"]]

    # convert list of bbox to columns
    bboxes = df["bbox"].apply(pd.Series, index=["col_x", "row_y", "width", "height"])

    # combine dataframes
    df = df.drop(columns=["bbox"]).join(bboxes, validate="1:1")

    df["label"] = "drone"
    return df


def prepare_annotations(root_dir: pathlib.Path):
    annotation_dir = root_dir / "annotations"

    result = pd.DataFrame()
    for subset, split in SPLIT_MAP.items():
        subset_file = annotation_dir / f"{subset}.json"
        if not subset_file.exists():
            warnings.warn(f"The subset {subset} is not found.")
            continue

        with open(subset_file, "r") as f:
            content = json.load(f)

        images = process_images(content)
        annotations = process_annotations(content)
        df = images.merge(annotations, on="image_id", validate="1:m")

        df["split"] = split
        df = df.drop(columns=["image_id"])

        result = pd.concat([result, df], ignore_index=True)
    return result

In [12]:
root_dir = pathlib.Path("~/data/UAV/DDS").expanduser()
work_dir = pathlib.Path("~/work/work_dirs/fastdup/dds").expanduser()

annotations = prepare_annotations(root_dir)
# annotation_dir = root_dir / "annotations"
# annotations = [annotation_dir / "train.json", annotation_dir]

fd = fastdup.create(input_dir=root_dir / "images", work_dir=work_dir)
fd.run(annotations=annotations)

/tmp/ipykernel_2102940/3047095884.py:32: UserWarning: The subset test is not found.
  warnings.warn(f"The subset {subset} is not found.")


FastDup Software, (C) copyright 2022 Dr. Amir Alush and Dr. Danny Bickson.
2024-06-02 22:35:29 [INFO] Going to loop over dir /tmp/tmps4ful5h0.csv
2024-06-02 22:35:29 [INFO] Found total 23873 images to run on, 23873 train, 0 test, name list 23873, counter 23873 
2024-06-02 22:35:35 [WARNING] Found invalid bounding box for image /home/leafying/data/UAV/DDS/images/00_06_10_to_00_06_27/00490.png. Please check bounding box file -5 1238 53 93
2024-06-02 22:40:16 [WARNING] Found invalid bounding box for image /home/leafying/data/UAV/DDS/images/2019_09_02_C0002_3700_mavic/00925.png. Please check bounding box file -3 2260 25 40
2024-06-02 22:41:02 [WARNING] Found invalid bounding box for image /home/leafying/data/UAV/DDS/images/2019_10_16_C0003_1700_matrice/00900.png. Please check bounding box file -9 1800 75 83
2024-06-02 22:41:34 [WARNING] Found invalid bounding box for image /home/leafying/data/UAV/DDS/images/2019_10_16_C0003_1700_matrice/01365.png. Please check bounding box file -11 1289 72

0

In [13]:
fd.vis.similarity_gallery()

Generating gallery:   0%|          | 0/20 [00:00<?, ?it/s]

Stored similar images visual view in  /home/leafying/work/work_dirs/fastdup/dds/galleries/similarity.html
########################################################################################
Would you like to see awesome visualizations for some of the most popular academic datasets?
Click here to see and learn more: https://app.visual-layer.com/vl-datasets?utm_source=fastdup
########################################################################################


,from,to,label,label2,distance
3902,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00175.png_1228_479_88_66.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00140.png_1102_476_85_65.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00180.png_1245_481_90_68.jpg]","[drone, drone]","[drone, drone]","[0.900038, 0.9145]"
2299,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesGOPR5842_00200055.png_1305_273_42_37.jpg,[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesGOPR5842_00200050.png_1334_257_39_36.jpg],[drone],[drone],[0.900046]
3962,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00490.png_1255_490_71_54.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00480.png_1275_508_71_52.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_matrice_210_hillside00465.png_1239_533_61_52.jpg]","[drone, drone]","[drone, drone]","[0.900085, 0.905329]"
1810,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_11_14_C0001_3922_matrice00055.png_1875_873_56_30.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_11_14_C0001_3922_matrice00065.png_2011_779_54_31.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_11_14_C0001_3922_matrice00050.png_1804_917_56_31.jpg]","[drone, drone]","[drone, drone]","[0.900087, 0.906884]"
6758,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_phantom_4_long_takeoff01090.png_844_711_18_13.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_phantom_4_long_takeoff00980.png_866_868_14_12.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimagesdji_phantom_4_long_takeoff01070.png_861_763_11_12.jpg]","[drone, drone]","[drone, drone]","[0.900118, 0.906741]"
...,...,...,...,...,...
1652,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00005.png_2133_983_183_78.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00000.png_2133_983_183_78.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00010.png_2133_983_183_78.jpg]","[drone, drone]","[drone, drone]","[0.999935, 1.0]"
1653,/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00010.png_2133_983_183_78.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00000.png_2133_983_183_78.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/0/homeleafyingdataUAVDDSimages2019_10_16_C0003_4613_mavic00005.png_2133_983_183_78.jpg]","[drone, drone]","[drone, drone]","[0.999935, 1.0]"
9817,/home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00005.png_688_454_13_11.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00010.png_688_454_13_11.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00000.png_688_454_13_11.jpg]","[drone, drone]","[drone, drone]","[1.0, 1.0]"
9816,/home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00000.png_688_454_13_11.jpg,"[/home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00010.png_688_454_13_11.jpg, /home/leafying/work/work_dirs/fastdup/dds/crops/1/homeleafyingdataUAVDDSimages2019_09_02_GOPR5871_1058_solo00005.png_688_454_13_11.jpg]","[dron

In [14]:
fd.vis.component_gallery(metric="size")

drone


Generating gallery:   0%|          | 0/20 [00:00<?, ?it/s]

Finished OK. Components are stored as image files /home/leafying/work/work_dirs/fastdup/dds/galleries/components_[index].jpg
Stored components visual view in  /home/leafying/work/work_dirs/fastdup/dds/galleries/components.html
Execution time in seconds 1.6
########################################################################################
Would you like to see awesome visualizations for some of the most popular academic datasets?
Click here to see and learn more: https://app.visual-layer.com/vl-datasets?utm_source=fastdup
########################################################################################


0

In [15]:
fd.vis.duplicates_gallery()

Generating gallery:   0%|          | 0/20 [00:00<?, ?it/s]

Stored similarity visual view in  /home/leafying/work/work_dirs/fastdup/dds/galleries/duplicates.html
########################################################################################
Would you like to see awesome visualizations for some of the most popular academic datasets?
Click here to see and learn more: https://app.visual-layer.com/vl-datasets?utm_source=fastdup
########################################################################################


0

In [16]:
fd.vis.outliers_gallery()

Generating gallery:   0%|          | 0/20 [00:00<?, ?it/s]

Stored outliers visual view in  /home/leafying/work/work_dirs/fastdup/dds/galleries/outliers.html
########################################################################################
Would you like to see awesome visualizations for some of the most popular academic datasets?
Click here to see and learn more: https://app.visual-layer.com/vl-datasets?utm_source=fastdup
########################################################################################


0

In [17]:
fd.invalid_instances()

,col_x,row_y,width,height,label,split,crop_filename,filename,index,is_valid
24,1879,290,56,46,drone,train,NaN,NaN,NaN,False
71,1238,-5,93,53,drone,train,NaN,NaN,NaN,False
255,410,735,17,9,drone,train,NaN,NaN,NaN,False
265,393,738,16,9,drone,train,NaN,NaN,NaN,False
266,391,739,16,8,drone,train,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...
21837,969,836,5,4,drone,val,NaN,NaN,NaN,False
22505,606,-5,21,16,drone,val,NaN,NaN,NaN,False
22560,240,-2,24,17,drone,val,NaN,NaN,NaN,False
22562,232,-6,21,15,drone,val,NaN,NaN,NaN,False
